# ChessLab Data Storage

This notebook demonstrates how ChessLab moves from Chess.com game data to structured data stored in a SQLite database.

## Workflow

1. Chess.com API
2. PGN
3. Parsed games
4. Game information
5. SQLite database
6. Queryable game data

In [34]:
from pathlib import Path

from chesslab.api.chesscom import download_monthly_games
from chesslab.chess.parser import parse_games, get_game_info
from chesslab.storage.database import (
    create_connection,
    create_games_table,
    save_game,
    get_games,
)

## 1. Select a player and month

As before, we first need to select a username and month

In [35]:
# we'll keep it the same, we'll use Sept 2026 and my own game data
username = "tjmoonz98"
year = 2026
month = 9

## 2. Download the games

First we need to retrieve the raw PGN data from Chess.com.

In [36]:
# define pgn object using input variables and print character stats
pgn = download_monthly_games(
    username,
    year,
    month,
)

print(f"Downloaded {len(pgn):,} characters of PGN data.")

Downloaded 180,892 characters of PGN data.


## 3. Parse games

In [37]:
# parse games with ChessLab functions
games = parse_games(pgn)

print(f"Games found: {len(games)}")

Games found: 79


## 4. Extract game information

Convert each `python-chess` Game object into a simple dictionary
containing the information we want to store, using a new parser function, `get_game_info`.

In [38]:
# define game info and print first game
game_info = [
    get_game_info(game, username)
    for game in games
]

print(game_info[0])

{'username': 'tjmoonz98', 'white': 'tjmoonz98', 'black': 'banban233', 'result': '1-0', 'date': '2026.09.15', 'eco': 'B20', 'white_elo': '805', 'black_elo': '138', 'time_control': '1/259200', 'termination': 'tjmoonz98 won by checkmate', 'link': 'https://www.chess.com/game/daily/1029824750'}


## 5. Create the database

The database will be stored locally in `data/chesslab.db`.

The database itself is ignored by Git and will not be committed to the repository, this is handled by the `.gitignore` file.

In [39]:
# create database in the following directory
database_path = Path("data/chesslab.db")
database_path.parent.mkdir(exist_ok=True)

# cteate the connection to said database
connection = create_connection(database_path)

# create table with connection and print path
create_games_table(connection)

print(f"Database created at: {database_path}")

Database created at: data/chesslab.db


## 6. Store the games

Now we can save each parsed game to SQLite.

For the first version of ChessLab, the original PGN is also retained so that we don't lose any information from the original source data (we may find that we need this later).

In [40]:
# save game info to the created database via the connection
for game, info in zip(games, game_info):
    save_game(
        connection,
        info,
        str(game),
    )

print(f"Stored {len(games)} games.")

Stored 79 games.


## 7. Retrieve the stored games

We will also need to fetch data from the created db at times.

In [41]:
# define stored games object using ChessLab's get_games function
stored_games = get_games(connection)

print(f"Games in database: {len(stored_games)}")

Games in database: 79


## 8. Inspect a stored game

Each row contains the metadata for one game followed by the original PGN.

In [42]:
# look at first game in stored games
stored_games[0]

(1,
 'tjmoonz98',
 'tjmoonz98',
 'banban233',
 '1-0',
 '2026.09.15',
 'B20',
 805,
 138,
 '1/259200',
 'tjmoonz98 won by checkmate',
 'https://www.chess.com/game/daily/1029824750',
 '[Event "Let\'s Play!"]\n[Site "Chess.com"]\n[Date "2026.09.15"]\n[Round "-"]\n[White "tjmoonz98"]\n[Black "banban233"]\n[Result "1-0"]\n[CurrentPosition "6rk/p7/2np3Q/qpp1p2P/4P1p1/1PPP2B1/P5P1/RN2KR2 b Q - 0 28"]\n[Timezone "UTC"]\n[ECO "B20"]\n[ECOUrl "https://www.chess.com/openings/Sicilian-Defense-2.d3"]\n[UTCDate "2026.09.15"]\n[UTCTime "22:38:27"]\n[WhiteElo "805"]\n[BlackElo "138"]\n[TimeControl "1/259200"]\n[Termination "tjmoonz98 won by checkmate"]\n[StartTime "22:38:27"]\n[EndDate "2026.09.28"]\n[EndTime "20:19:46"]\n[Link "https://www.chess.com/game/daily/1029824750"]\n\n1. e4 { [%clk 0:00:00.5] } 1... c5 { [%clk 1:04:58.6] } 2. d3 { [%clk 0:01:19.7] } 2... e5 { [%clk 0:00:36.2] } 3. Nf3 { [%clk 0:00:19] } 3... d6 { [%clk 0:00:01.3] } 4. Bg5 { [%clk 0:00:05.2] } 4... f6 { [%clk 0:00:05] } 5. Bh4

## 9. Query the database

Because the data is now stored in our SQLite database, we can use SQL to answer questions about the player's games.

In [50]:
# use SQL to fetch data, grouped by outcome
cursor = connection.execute(
    """
    SELECT termination, COUNT(*) AS game_count
    FROM games
    GROUP BY termination
    ORDER BY game_count DESC
    """
)

results = cursor.fetchall()

results


[('tjmoonz98 won by checkmate', 20),
 ('tjmoonz98 won by resignation', 14),
 ('tjmoonz98 won - game abandoned', 5),
 ('tjmoonz98 won on time', 4),
 ('Game drawn by stalemate', 2),
 ('zoubir64 won by checkmate', 1),
 ('virendus won by resignation', 1),
 ('tinetoon8 won by resignation', 1),
 ('the_disgraced_one99 won by checkmate', 1),
 ('skr_x won by checkmate', 1),
 ('pumcia1104 won by resignation', 1),
 ('mayennaisenforce won by checkmate', 1),
 ('matty2blunders won by checkmate', 1),
 ('le-nox won by checkmate', 1),
 ('later_at_night won by checkmate', 1),
 ('juvanjansevanrensburg won by resignation', 1),
 ('juliussaraspi won by resignation', 1),
 ('hosamgab won by resignation', 1),
 ('dfb87 won by checkmate', 1),
 ('chess_master1913 won by resignation', 1),
 ('bleahhhh21 won by checkmate', 1),
 ('amansonkar83 won by resignation', 1),
 ('aimadsabbani won by resignation', 1),
 ('ZHSereno won by resignation', 1),
 ('Vijayanagase won by resignation', 1),
 ('TMBADGAMER12 won on time', 1)

## 10. Games by colour

We can also use SQL to investigate how often a player played White or Black.

In [ ]:
# retrieve all counts for white games, order descending - note: I have played White 40 times out of the 79
cursor = connection.execute(
    """
    SELECT white, COUNT(*) AS game_count
    FROM games
    GROUP BY white
    ORDER BY game_count DESC
    """
)

cursor.fetchall()

[('tjmoonz98', 40),
 ('zoubir64', 1),
 ('yasston94', 1),
 ('virendus', 1),
 ('unknown_01245', 1),
 ('the_disgraced_one99', 1),
 ('skr_x', 1),
 ('matty2blunders', 1),
 ('le-nox', 1),
 ('later_at_night', 1),
 ('juliussaraspi', 1),
 ('infinitely10', 1),
 ('dfb87', 1),
 ('brownbambi', 1),
 ('bleahhhh21', 1),
 ('bennyphilip18', 1),
 ('banban233', 1),
 ('aimadsabbani', 1),
 ('ZHSereno', 1),
 ('Vijayanagase', 1),
 ('TedClifton', 1),
 ('TMBADGAMER12', 1),
 ('T1Abyss', 1),
 ('Sandeepsawna', 1),
 ('RedHotChiliPepper2023', 1),
 ('Pipat_Pat', 1),
 ('Mohamed1983103', 1),
 ('Klobb25', 1),
 ('Jovapo1313', 1),
 ('GrandMastrAds', 1),
 ('ElPinichini', 1),
 ('Eishaazher', 1),
 ('Didier-66', 1),
 ('Dalia-33', 1),
 ('Biggestswae', 1),
 ('Anwarkutty', 1),
 ('Amiga1984', 1),
 ('AlvisFreimanis', 1),
 ('AMER_AL_SHOUFI', 1),
 ('9diyar9', 1)]

## 11. Closing the database

We are now finished with our database connection for this notebook.

In [51]:
# close connection
connection.close()

print("Database connection closed.")

Database connection closed.


## What We've Learned

In this notebook, we:

* Connected our parsed chess games to a SQLite database.
* Designed and created a simple `games` table.
* Stored both game metadata and the original PGN.
* Retrieved games from the database using SQL queries.

This gives ChessLab a foundation for the next stages of the project, where we can begin analysing games with chess engines and calculating player-level statistics.